# Carga y Paths #

In [746]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RUTA_NOTEBOOK = Path.cwd()
BASE_DIR = RUTA_NOTEBOOK.parent

DIR_RAW = BASE_DIR / "data" / "raw"
DIR_PROCESSED = BASE_DIR / "data" / "processed"

PATH_ENTRADA = BASE_DIR / "data" / "raw" / "datos_raw_cafeteria.csv"
PATH_SALIDA = BASE_DIR / "data" / "processed" / "datos_clean_cafeteria.csv"

DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto: {BASE_DIR}")
print(f"Buscando en: {PATH_ENTRADA}")
print(f"Guadando salida en: {PATH_SALIDA}") 

Raiz del proyecto: c:\Users\OAD\Documents\pyme-dashboards-cafeteria
Buscando en: c:\Users\OAD\Documents\pyme-dashboards-cafeteria\data\raw\datos_raw_cafeteria.csv
Guadando salida en: c:\Users\OAD\Documents\pyme-dashboards-cafeteria\data\processed\datos_clean_cafeteria.csv


In [747]:
df = pd.read_csv(PATH_ENTRADA)
df

,ID_Ticket,Fecha_Hora,RUT_Cliente,Mesa,Producto_Categoria,Producto_Detail,Cantidad,Precio_Unitario_IVA,Metodo_Pago
0,TX-20000,01/01/2025 09:06,98765432-1,Mesa 3,Bebestibles,JUGO ARANDANO,3.0,3100,Débito
1,TX-20000,01/01/2025 09:09,98765432-1,Mesa 3,Cafe,TE NEGRO,NaN,2100,Débito
2,TX-20000,2025-01-01 09:12:07,98765432-1,Mesa 3,Repostería,muffin vainilla,2.0,2100,Débito
3,TX-20001,2025-01-01 09:20:48,Consumidor Final,Mesa 6,cafes,espresso doble,2.0,$ 2.490,Crédito
4,TX-20001,2025-01-01 09:22:47,Consumidor Final,Mesa 6,Bebestible,Jugo Naranja Natural,1.0,2990,Crédito
...,...,...,...,...,...,...,...,...,...
13729,TX-24497,46022.8162,Consumidor Final,Mesa 7,Bebestible,Jugo de Piña,1.0,2990,Débito
13730,TX-24498,31/12/2025 19:36,Consumidor Final,Mesa 2,Repostería,medialuna,1.0,$ 1.400,Débito
13731,TX-24498,2025-12-31 19:39:13,Consumidor Final,Mesa 2,CAFÉ,Flat White,1.0,3000,Débito
13732,TX-24499,2025-12-31 19:40:25,Consumidor Final,Mesa 6,Cafés,cafe americano,3.0,2200,Débito


*Inspección inicial.*

In [748]:
df.shape

(13734, 9)

In [749]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13734 entries, 0 to 13733
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_Ticket            13734 non-null  object 
 1   Fecha_Hora           13734 non-null  object 
 2   RUT_Cliente          13734 non-null  object 
 3   Mesa                 13734 non-null  object 
 4   Producto_Categoria   13734 non-null  object 
 5   Producto_Detail      13734 non-null  object 
 6   Cantidad             13458 non-null  float64
 7   Precio_Unitario_IVA  13734 non-null  object 
 8   Metodo_Pago          13734 non-null  object 
dtypes: float64(1), object(8)
memory usage: 965.8+ KB


In [750]:
df.describe()

,Cantidad
count,13458.000000
mean,1.611235
std,0.808018
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,3.000000


# Limpieza. #

**Duplicados.**

In [751]:
df.duplicated().sum().sum()

np.int64(280)

In [752]:
df[df.duplicated(keep=False)]

,ID_Ticket,Fecha_Hora,RUT_Cliente,Mesa,Producto_Categoria,Producto_Detail,Cantidad,Precio_Unitario_IVA,Metodo_Pago
53,TX-20017,2025-01-02 10:28:00,Consumidor Final,Mesa 9,Cafés,Flatwhite,1.0,3000,Crédito
54,TX-20017,2025-01-02 10:28:00,Consumidor Final,Mesa 9,Cafés,Flatwhite,1.0,3000,Crédito
106,TX-20035,2025-01-03 19:15:49,Consumidor Final,Mesa 8,reposteria,Muffin de Vainilla Natural,2.0,2100,Crédito
107,TX-20035,2025-01-03 19:15:49,Consumidor Final,Mesa 8,reposteria,Muffin de Vainilla Natural,2.0,2100,Crédito
153,TX-20050,2025-01-05 09:12:23,Consumidor Final,Mesa 1,PASTELERIA,medialuna manjar,3.0,$ 1.600,Débito
...,...,...,...,...,...,...,...,...,...
13539,TX-24420,2025-12-27 16:27:39,Consumidor Final,Mesa 7,bebestibles,Jugo Arándanos,3.0,3100,Débito
13546,TX-24422,27/12/2025 18:36,Consumidor Final,Para Llevar,Sandwichería,croissant caprese,3.0,4790,Débito
13547,TX-24422,27/12/2025 18:36,Consumidor Final,Para Llevar,Sandwichería,croissant caprese,3.0,4790,Débito
13558,TX-24426,2025-12-28 09:33:55,Consumidor Final,Mesa 8,Sandwichería,Croissant Jamón Queso,1.0,$ 4.500,Débito


In [753]:
df = df.drop_duplicates()
print("Duplicados actuales: ", df.duplicated().sum())
print("Filas y Columnas: ", df.shape)

Duplicados actuales:  0
Filas y Columnas:  (13454, 9)


*Todos los duplicados perfectos descartados.*

**Nulos.**

In [754]:
df.isna().sum()

ID_Ticket                0
Fecha_Hora               0
RUT_Cliente              0
Mesa                     0
Producto_Categoria       0
Producto_Detail          0
Cantidad               272
Precio_Unitario_IVA      0
Metodo_Pago              0
dtype: int64

In [755]:
df[df.isna().any(axis=1)]

,ID_Ticket,Fecha_Hora,RUT_Cliente,Mesa,Producto_Categoria,Producto_Detail,Cantidad,Precio_Unitario_IVA,Metodo_Pago
1,TX-20000,01/01/2025 09:09,98765432-1,Mesa 3,Cafe,TE NEGRO,NaN,2100,Débito
16,TX-20005,01/01/2025 10:25,Consumidor Final,Mesa 7,cafes,Cafe Late,NaN,3200,Efectivo
38,TX-20012,2025-01-01 19:55:41,Consumidor Final,Mesa 10,Bebestible,JUGO NARANJA,NaN,2990,Crédito
244,TX-20080,2025-01-07 12:42:59,21445887-5,Mesa 5,Sandwichería,sandwich mechada queso,NaN,$ 5.490,Débito
276,TX-20090,2025-01-08 19:21:38,Consumidor Final,Para Llevar,cafes,TE NEGRO,NaN,$ 2.100,Débito
...,...,...,...,...,...,...,...,...,...
13575,TX-24431,2025-12-28 13:03:05,Consumidor Final,Mesa 6,Sandwichería,croissant mechada,NaN,5690,Débito
13591,TX-24437,2025-12-28 19:02:15,Consumidor Final,Mesa 3,bebestibles,leche platano,NaN,2700,Débito
13604,TX-24441,2025-12-29 09:09:27,Consumidor Final,Para Llevar,reposteria,muffin arandano,NaN,2100,Débito
13635,TX-24451,29/12/2025 18:08,16443882-1,Mesa 5,cafes,Mocaccino,NaN,3400,Débito


In [756]:
df.isna().mean() * 100

ID_Ticket              0.000000
Fecha_Hora             0.000000
RUT_Cliente            0.000000
Mesa                   0.000000
Producto_Categoria     0.000000
Producto_Detail        0.000000
Cantidad               2.021704
Precio_Unitario_IVA    0.000000
Metodo_Pago            0.000000
dtype: float64

*Los nulos, que están en una columna clave ("Cantidad"), son utiles y necesarios. Sin embargo, al ser menos de un 5% los descartamos.*

In [757]:
df = df.dropna()
print("Nulos actuales: ", df.isna().sum())
print("Filas y Columnas: ", df.shape)

Nulos actuales:  ID_Ticket              0
Fecha_Hora             0
RUT_Cliente            0
Mesa                   0
Producto_Categoria     0
Producto_Detail        0
Cantidad               0
Precio_Unitario_IVA    0
Metodo_Pago            0
dtype: int64
Filas y Columnas:  (13182, 9)


**Dtypes.**

*Partimos por dtypes "object":*

In [758]:
df.dtypes

ID_Ticket               object
Fecha_Hora              object
RUT_Cliente             object
Mesa                    object
Producto_Categoria      object
Producto_Detail         object
Cantidad               float64
Precio_Unitario_IVA     object
Metodo_Pago             object
dtype: object

In [759]:
df["Cantidad"] = df["Cantidad"].astype(int) #"Cantidad" a numero entero.

In [760]:
df["Precio_Unitario_IVA"] = df["Precio_Unitario_IVA"].astype(str).str.replace(r"[$\s.]", "", regex=True) #Eliminamos simbolos de "Precio_Unitario_IVA"
df["Precio_Unitario_IVA"] = df["Precio_Unitario_IVA"].astype(int) #"Precio_Unitario_IVA" a numero entero.

In [761]:
df["Metodo_Pago"].value_counts()

Metodo_Pago
Débito           7998
Crédito          3303
Efectivo         1223
Transferencia     658
Name: count, dtype: int64

**"Metodo_Pago" se mantiene.**

In [762]:
df["Mesa"].value_counts()

Mesa
Para Llevar    3098
Mesa 6         1588
Mesa 5         1528
Mesa 7         1494
Mesa 8         1482
Mesa 2          795
Mesa 1          715
Mesa 4          711
Mesa 9          645
Mesa 3          606
Mesa 10         520
Name: count, dtype: int64

In [763]:
df["Mesa"] = df["Mesa"].replace("Para Llevar", 0) #"Para Llevar" = 0

In [764]:
df["Numero de mesa"] = df["Mesa"].astype(str).str.extract(r"(\d+)") #Extraer numero desde el string.
df["Numero de mesa"] = df["Numero de mesa"].astype(int) #"Numero de mesa" a numero entero.

In [765]:
df["Tipo de servicio"] = np.where( 
    df["Numero de mesa"] == 0, "Para llevar",
    "Para servir"
) #Numero de mesa 0 es igual a "Para llevar", todo lo demas: "Para servir"

In [766]:
df = df.drop(columns="Mesa") #Descartamos la columna "Mesa" ya que tenemos su desglose.

**"Producto_Categoria" y "Producto_Detail".**

In [767]:
df["Producto_Categoria"] = df["Producto_Categoria"].astype(str)
print(f"Hay {df["Producto_Categoria"].nunique()} tipos de variables")
print(df["Producto_Categoria"].unique())
df["Producto_Categoria"] = df["Producto_Categoria"].str.strip()

Hay 16 tipos de variables
['Bebestibles' 'Repostería' 'cafes' 'Bebestible' 'sandwicheria' 'Cafe'
 'CAFÉ' 'Cafés' 'bebestibles' 'Sandwichería' 'Reposteria ' 'SANDWICH'
 'BEBIDAS' 'PASTELERIA' 'Sándwiches' 'reposteria']


In [768]:
valores_antiguos_cafe = r"Cafés|CAFÉ|cafes|Cafe"
df["Producto_Categoria"] = (df["Producto_Categoria"].str.replace(valores_antiguos_cafe, "Cafés", case=False, regex=True))
valores_antiguos_reposteria = r"PASTELERIA|Reposteria|reposteria|Repostería"
df["Producto_Categoria"] = (df["Producto_Categoria"].str.replace(valores_antiguos_reposteria, "Repostería", case=False, regex=True))
valores_antiguos_bebestible = r"BEBIDAS|bebestibles|Bebestibles|Bebestible"
df["Producto_Categoria"] = (df["Producto_Categoria"].str.replace(valores_antiguos_bebestible, "Bebestibles", case=False, regex=True))
valores_antiguos_sandwiches = r"Sandwichería|sandwicheria|SANDWICH|Sándwiches"
df["Producto_Categoria"] = (df["Producto_Categoria"].str.replace(valores_antiguos_sandwiches, "Sandwicheria", case=False, regex=True))


In [769]:
df["Producto_Categoria"].unique()

array(['Bebestibles', 'Repostería', 'Cafés', 'Sandwicheria'], dtype=object)

In [770]:
df["Producto_Categoria"].value_counts()

Producto_Categoria
Cafés           4596
Repostería      3083
Sandwicheria    3050
Bebestibles     2453
Name: count, dtype: int64

In [771]:
df["Producto_Detail"] = df["Producto_Detail"].astype(str)
print(f"Hay {df["Producto_Detail"].nunique()} tipos de variables")
print(df["Producto_Detail"].unique())
df["Producto_Detail"] = df["Producto_Detail"].str.strip()

Hay 89 tipos de variables
['JUGO ARANDANO' 'muffin vainilla' 'espresso doble' 'Jugo Naranja Natural'
 'CROISSANT CAPRESE' 'TE VERDE' 'Sandwich Caprese Ciabatta' 'TE BLANCO'
 'te negro' 'jugo pina' 'cafe americano' 'Sándwich Mechada Queso'
 'Croissant J/Q' 'Muffin de Arándanos' 'Jugo Arándanos' 'te verde'
 'Capuccino' 'Americano' 'Aliado' 'matcha latte' 'Flatwhite'
 'waffle manjar' 'Té Blanco' 'Medialuna Dulce' 'sandwich caprese'
 'cafe latte ' 'TE NEGRO' 'medialuna manjar' 'Muffin de Vainilla Natural'
 'Flat White' 'Mechada Queso' 'sandwich mechada queso' 'Medialunas Manjar'
 'Leche con Plátano' 'Té Negro' 'Medialuna Dulce ' 'Té Matcha Latte'
 'CAFÉ LATTE' 'donas' 'CROISSANT MECHADA' 'Croissant Mechada Queso'
 'croissant caprese' 'MOCA' 'MUFFIN VAINILLA' 'Leche Plátano'
 'Sándwich Caprese' 'Té Verde' 'croissant mechada' 'Sándwich Jamón-Queso'
 'Capu ' 'Jugo de Piña' 'dona rellena' 'croissant jq' 'Café Latte'
 'Muffin Arandanos ' 'FLAT WHITE' 'medialuna' 'JUGO PIÑA' 'te blanco'
 'jugo a

In [772]:
df["Producto_Detail"] = df["Producto_Detail"].str.lower()
df["Producto_Detail"] = df["Producto_Detail"].str.strip()

In [773]:
pd.set_option('display.max_colwidth', None)
df.groupby("Producto_Categoria")["Producto_Detail"].unique()

Producto_Categoria
Bebestibles                                                                                                               [jugo arandano, jugo naranja natural, jugo pina, jugo arándanos, matcha latte, leche con plátano, té matcha latte, leche plátano, jugo de piña, jugo piña, jugo de arándano, jugo naranja, leche platano]
Cafés                                                                                   [espresso doble, te verde, te blanco, te negro, cafe americano, capuccino, americano, flatwhite, té blanco, cafe latte, flat white, té negro, café latte, moca, té verde, capu, cafe late, mochaccino, mocaccino, espresso dbl, cappuccino]
Repostería                                                                [muffin vainilla, muffin de arándanos, waffle manjar, medialuna dulce, medialuna manjar, muffin de vainilla natural, medialunas manjar, donas, dona rellena, muffin arandanos, medialuna, donas rellenas, medialunas, waffle con manjar, muffin arandano]
Sandwiche

In [774]:
df["Producto_Detail_Limpio"] = df["Producto_Detail"].astype(str).str.lower().str.strip()

mapa_estandarizado = {
    # Bebestibles
    "jugo pina": "Jugo de Piña",
    "jugo de piña": "Jugo de Piña",
    "jugo piña": "Jugo de Piña",
    "matcha latte": "Té Matcha Latte",
    "té matcha latte": "Té Matcha Latte",
    "jugo naranja": "Jugo Naranja Natural",
    "jugo naranja natural": "Jugo Naranja Natural",
    "jugo arandano": "Jugo de Arándano",
    "jugo de arándano": "Jugo de Arándano",
    "jugo arándanos": "Jugo de Arándano",
    "leche platano": "Leche con Plátano",
    "leche con plátano": "Leche con Plátano",
    "leche plátano": "Leche con Plátano",
    # Cafés
    "cafe latte": "Café Latte",
    "café latte": "Café Latte",
    "cafe late": "Café Latte",
    "té blanco": "Té Blanco",
    "te blanco": "Té Blanco",
    "americano": "Americano",
    "cafe americano": "Americano",
    "cappuccino": "Capuccino",
    "capuccino": "Capuccino",
    "capu": "Capuccino",
    "te negro": "Té Negro",
    "té negro": "Té Negro",
    "flat white": "Flat White",
    "flatwhite": "Flat White",
    "espresso doble": "Espresso Doble",
    "espresso dbl": "Espresso Doble",
    "te verde": "Té Verde",
    "té verde": "Té Verde",
    "mocaccino": "Mocaccino",
    "moca": "Mocaccino",
    "mochaccino": "Mocaccino",
    # Repostería
    "medialuna dulce": "Medialuna Dulce",
    "medialuna": "Medialuna Dulce",
    "medialunas": "Medialuna Dulce",
    "muffin arandanos": "Muffin de Arándanos",
    "muffin arandano": "Muffin de Arándanos",
    "muffin de arándanos": "Muffin de Arándanos",
    "waffle manjar": "Waffle con Manjar",
    "waffle con manjar": "Waffle con Manjar",
    "donas rellenas": "Donas Rellenas",
    "dona rellena": "Donas Rellenas",
    "donas": "Donas Rellenas",
    "muffin de vainilla natural": "Muffin de Vainilla Natural",
    "muffin vainilla": "Muffin de Vainilla Natural",
    "medialuna manjar": "Medialuna Manjar",
    "medialunas manjar": "Medialuna Manjar",
    # Sandwichería
    "sandwich mechada queso": "Sándwich Mechada Queso",
    "mechada queso": "Sándwich Mechada Queso",
    "sándwich mechada queso": "Sándwich Mechada Queso",
    "sandwich caprese ciabatta": "Sandwich Caprese Ciabatta",
    "sándwich caprese": "Sandwich Caprese Ciabatta",
    "sandwich caprese": "Sandwich Caprese Ciabatta",
    "croissant jq": "Croissant Jamón Queso",
    "croissant jamón queso": "Croissant Jamón Queso",
    "croissant caprese": "Croissant Caprese",
    "sandwich jamon queso": "Sándwich Jamón-Queso",
    "aliado": "Sándwich Jamón-Queso",
    "sándwich jamón-queso": "Sándwich Jamón-Queso",
    "sándwich j/q": "Sándwich Jamón-Queso",
    "croissant j/q": "Croissant Jamón Queso",
    "croissant mechada": "Croissant Mechada Queso",
    "croissant mechada queso": "Croissant Mechada Queso",
}

df["Producto_Detail"] = df["Producto_Detail_Limpio"].replace(mapa_estandarizado)
df.drop(columns=["Producto_Detail_Limpio"], inplace=True)

df.groupby("Producto_Categoria")["Producto_Detail"].unique()

Producto_Categoria
Bebestibles                                                       [Jugo de Arándano, Jugo Naranja Natural, Jugo de Piña, Té Matcha Latte, Leche con Plátano]
Cafés                                               [Espresso Doble, Té Verde, Té Blanco, Té Negro, Americano, Capuccino, Flat White, Café Latte, Mocaccino]
Repostería                           [Muffin de Vainilla Natural, Muffin de Arándanos, Waffle con Manjar, Medialuna Dulce, Medialuna Manjar, Donas Rellenas]
Sandwicheria    [Croissant Caprese, Sandwich Caprese Ciabatta, Sándwich Mechada Queso, Croissant Jamón Queso, Sándwich Jamón-Queso, Croissant Mechada Queso]
Name: Producto_Detail, dtype: object

In [775]:
pd.set_option('display.max_colwidth', None)
df.groupby("Producto_Categoria")["Producto_Detail"].unique()

Producto_Categoria
Bebestibles                                                       [Jugo de Arándano, Jugo Naranja Natural, Jugo de Piña, Té Matcha Latte, Leche con Plátano]
Cafés                                               [Espresso Doble, Té Verde, Té Blanco, Té Negro, Americano, Capuccino, Flat White, Café Latte, Mocaccino]
Repostería                           [Muffin de Vainilla Natural, Muffin de Arándanos, Waffle con Manjar, Medialuna Dulce, Medialuna Manjar, Donas Rellenas]
Sandwicheria    [Croissant Caprese, Sandwich Caprese Ciabatta, Sándwich Mechada Queso, Croissant Jamón Queso, Sándwich Jamón-Queso, Croissant Mechada Queso]
Name: Producto_Detail, dtype: object

In [776]:
df.loc[df["Producto_Detail"] == "Té Matcha Latte", "Producto_Categoria"] = "Cafés"
df["Producto_Detail"] = df["Producto_Detail"].str.title()

In [777]:
df.groupby("Producto_Categoria")["Producto_Detail"].unique()

Producto_Categoria
Bebestibles                                                                        [Jugo De Arándano, Jugo Naranja Natural, Jugo De Piña, Leche Con Plátano]
Cafés                              [Espresso Doble, Té Verde, Té Blanco, Té Negro, Americano, Capuccino, Té Matcha Latte, Flat White, Café Latte, Mocaccino]
Repostería                           [Muffin De Vainilla Natural, Muffin De Arándanos, Waffle Con Manjar, Medialuna Dulce, Medialuna Manjar, Donas Rellenas]
Sandwicheria    [Croissant Caprese, Sandwich Caprese Ciabatta, Sándwich Mechada Queso, Croissant Jamón Queso, Sándwich Jamón-Queso, Croissant Mechada Queso]
Name: Producto_Detail, dtype: object

**"RUT_Cliente"**

In [778]:
df["RUT_Cliente"] = df["RUT_Cliente"].str.strip()
print(df["RUT_Cliente"])

0              98765432-1
2              98765432-1
3        Consumidor Final
4        Consumidor Final
5        Consumidor Final
               ...       
13729    Consumidor Final
13730    Consumidor Final
13731    Consumidor Final
13732    Consumidor Final
13733    Consumidor Final
Name: RUT_Cliente, Length: 13182, dtype: object


In [779]:
df["RUT_Cliente"] = (df["RUT_Cliente"].astype(str).str.replace(".", "", regex=False).str.replace("-", "", regex=False))
df["RUT_Cliente"] = df["RUT_Cliente"].str.upper()

In [780]:
print(df["RUT_Cliente"])

0               987654321
2               987654321
3        CONSUMIDOR FINAL
4        CONSUMIDOR FINAL
5        CONSUMIDOR FINAL
               ...       
13729    CONSUMIDOR FINAL
13730    CONSUMIDOR FINAL
13731    CONSUMIDOR FINAL
13732    CONSUMIDOR FINAL
13733    CONSUMIDOR FINAL
Name: RUT_Cliente, Length: 13182, dtype: object


**Desglosamos "Fecha_Hora"**

In [781]:
df["Fecha_Hora"].astype(str).str.replace(r"\d", "d", regex=True).str.replace(r"[a-zA-Z]", "a", regex=True).value_counts()

Fecha_Hora
aaaa-aa-aa aa:aa:aa    8616
aa/aa/aaaa aa:aa       2606
aaaaa.aaaa             1761
aaaaa.aaa               182
aaaaa.aa                 17
Name: count, dtype: int64

In [782]:
df_fechas_raras = df[pd.to_datetime(df["Fecha_Hora"], format="mixed", errors="coerce").isna()]
print(df_fechas_raras["Fecha_Hora"].unique())

['45658.4199' '45658.4251' '45658.651' ... '46022.5467' '46022.7612'
 '46022.8162']


In [783]:
fechas_num = pd.to_numeric(df["Fecha_Hora"], errors="coerce")
df["Fecha_Hora_Limpia"] = pd.to_datetime(fechas_num, unit="D", origin="1899-12-30").fillna(
    pd.to_datetime(df["Fecha_Hora"], format="mixed", dayfirst=True, errors="coerce")
)

In [784]:
df["Fecha_Hora_Limpia"].astype(str).str.replace(r"\d", "d", regex=True).str.replace(r"[a-zA-Z]", "a", regex=True).value_counts()

Fecha_Hora_Limpia
aaaa-aa-aa aa:aa:aa.aaaaaaaaa    13182
Name: count, dtype: int64

In [785]:
df["Fecha_Hora"] = df["Fecha_Hora_Limpia"]

**"ID_Ticket"**

In [786]:
df["ID_Ticket"] = df["ID_Ticket"].str.strip()
df["ID_Ticket"] = df["ID_Ticket"].str.upper()

# Información Adicional #

**Coste de productos**

In [787]:
mapa_costes = {
    # Bebestibles
    "jugo de piña": 900,
    "jugo piña": 900,
    "jugo naranja natural": 900,
    "jugo de arándano": 950,
    "jugo de arandano": 950,
    "leche con plátano": 800,
    "leche con platano": 800,
    # Cafés e Infusiones
    "café latte": 850,
    "cafe latte": 850,
    "té blanco": 450,
    "te blanco": 450,
    "americano": 500,
    "capuccino": 800,
    "cappuccino": 800,
    "té matcha latte": 950,
    "te matcha latte": 950,
    "té negro": 400,
    "te negro": 400,
    "flat white": 800,
    "espresso doble": 600,
    "té verde": 400,
    "te verde": 400,
    "mocaccino": 900,
    # Repostería
    "medialuna dulce": 350,
    "muffin de arándanos": 600,
    "muffin de arandanos": 600,
    "waffle con manjar": 900,
    "donas rellenas": 600,
    "muffin de vainilla natural": 550,
    "medialuna manjar": 400,
    # Sandwichería
    "sándwich mechada queso": 1800,
    "sandwich mechada queso": 1800,
    "sandwich caprese ciabatta": 1600,
    "sándwich caprese ciabatta": 1600,
    "croissant jamón queso": 1400,
    "croissant jamon queso": 1400,
    "croissant caprese": 1500,
    "sándwich jamón-queso": 1200,
    "sandwich jamon-queso": 1200,
    "sandwich jamon queso": 1200,
    "croissant mechada queso": 1900,
}
df["Coste_Producto"] = (
    df["Producto_Detail"]
    .str.lower()
    .str.strip()
    .map(mapa_costes)
    .fillna(500)
    .astype(int)
)

In [788]:
df["Precio_Unitario_Neto"] = (df["Precio_Unitario_IVA"] / 1.19).round()

In [789]:
columnas_ordenadas = ["ID_Ticket", "RUT_Cliente", "Fecha_Hora", "Producto_Categoria", "Producto_Detail", "Cantidad", "Precio_Unitario_IVA", "Precio_Unitario_Neto", "Coste_Producto", "Metodo_Pago", "Numero de mesa", "Tipo de servicio"]
df = df[columnas_ordenadas]
df["Precio_Unitario_Neto"] = df["Precio_Unitario_Neto"].astype(int)

---

# Tabla limpia #

In [790]:
df

,ID_Ticket,RUT_Cliente,Fecha_Hora,Producto_Categoria,Producto_Detail,Cantidad,Precio_Unitario_IVA,Precio_Unitario_Neto,Coste_Producto,Metodo_Pago,Numero de mesa,Tipo de servicio
0,TX-20000,987654321,2025-01-01 09:06:00.000000000,Bebestibles,Jugo De Arándano,3,3100,2605,950,Débito,3,Para servir
2,TX-20000,987654321,2025-01-01 09:12:07.000000000,Repostería,Muffin De Vainilla Natural,2,2100,1765,550,Débito,3,Para servir
3,TX-20001,CONSUMIDOR FINAL,2025-01-01 09:20:48.000000000,Cafés,Espresso Doble,2,2490,2092,600,Crédito,6,Para servir
4,TX-20001,CONSUMIDOR FINAL,2025-01-01 09:22:47.000000000,Bebestibles,Jugo Naranja Natural,1,2990,2513,900,Crédito,6,Para servir
5,TX-20001,CONSUMIDOR FINAL,2025-01-01 09:27:58.000000000,Sandwicheria,Croissant Caprese,1,4790,4025,1500,Crédito,6,Para servir
...,...,...,...,...,...,...,...,...,...,...,...,...
13729,TX-24497,CONSUMIDOR FINAL,2025-12-31 19:35:19.680000095,Bebestibles,Jugo De Piña,1,2990,2513,900,Débito,7,Para servir
13730,TX-24498,CONSUMIDOR FINAL,2025-12-31 19:36:00.000000000,Repostería,Medialuna Dulce,1,1400,1176,350,Débito,2,Para servir
13731,TX-24498,CONSUMIDOR FINAL,2025-12-31 19:39:13.000000000,Cafés,Flat White,1,3000,2521,800,Débito,2,Para servir
13732,TX-24499,CONSUMIDOR FINAL,2025-12-31 19:40:25.000000000,Cafés,Americano,3,2200,1849,500,Débito,6,Para servir


In [791]:
df.shape

(13182, 12)

In [792]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13182 entries, 0 to 13733
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID_Ticket             13182 non-null  object        
 1   RUT_Cliente           13182 non-null  object        
 2   Fecha_Hora            13182 non-null  datetime64[ns]
 3   Producto_Categoria    13182 non-null  object        
 4   Producto_Detail       13182 non-null  object        
 5   Cantidad              13182 non-null  int64         
 6   Precio_Unitario_IVA   13182 non-null  int64         
 7   Precio_Unitario_Neto  13182 non-null  int64         
 8   Coste_Producto        13182 non-null  int64         
 9   Metodo_Pago           13182 non-null  object        
 10  Numero de mesa        13182 non-null  int64         
 11  Tipo de servicio      13182 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(6)
memory usage: 1.3+ MB


In [793]:
df.dtypes

ID_Ticket                       object
RUT_Cliente                     object
Fecha_Hora              datetime64[ns]
Producto_Categoria              object
Producto_Detail                 object
Cantidad                         int64
Precio_Unitario_IVA              int64
Precio_Unitario_Neto             int64
Coste_Producto                   int64
Metodo_Pago                     object
Numero de mesa                   int64
Tipo de servicio                object
dtype: object

In [794]:
df.isna().sum().sum()

np.int64(0)

---

# Export #

In [795]:
df.to_csv(PATH_SALIDA, index=False)